# Chapter 5: Data Processing and Transformation

This notebook complements Chapter 5 by turning processing design into a reproducibility and semantic-correctness exercise.

Use the retail ranking track for incremental features, label windows, and replay boundaries. Use the support-assistant track for chunking, embeddings, benchmark corpora, and prompt assembly.

The updated chapter adds one explicit runtime distinction:
- prompt template: reusable instruction pattern
- prompt instance: exact runtime input for one request
- context: retrieved or supplied information added to the prompt instance
- trace: durable execution record
- evaluation result: judgment of the final behavior

In [ ]:
from pathlib import Path

def find_companion_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "datasets").exists() and (candidate / "code").exists():
            return candidate
    raise FileNotFoundError("Could not find companion repository root.")


def ensure_path_exists(path: Path, kind: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing expected {kind}: {path}. "
            "Make sure you are running this notebook from inside the companion repository."
        )
    return path

NOTEBOOK_DIR = Path.cwd()
COMPANION_ROOT = find_companion_root(NOTEBOOK_DIR)
print("Companion root located successfully.")

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location

import pandas as pd

module_path = ensure_path_exists(
    COMPANION_ROOT / "code" / "processing" / "pipeline_planning.py",
    "helper module",
)
spec = spec_from_file_location("pipeline_planning", module_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load helper module spec from {module_path}")
pipeline_planning = module_from_spec(spec)
spec.loader.exec_module(pipeline_planning)

load_processing_scenarios = pipeline_planning.load_processing_scenarios
summarize_processing_pressure = pipeline_planning.summarize_processing_pressure
build_processing_plan = pipeline_planning.build_processing_plan
find_high_risk_pipelines = pipeline_planning.find_high_risk_pipelines

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

### Why the Setup Code Matters

The helper module is there to keep the notebook centered on processing contracts: replay strategy, incremental boundaries, evaluation leakage risk, and retrieval or prompt-assembly transforms. The important output is not the dataframe itself. It is the recommendation logic attached to each pipeline shape.

## Part 1: Load Processing Scenarios

The scenario table now includes benchmark rebuilds and retrieval-preparation stages in addition to classic batch and streaming pipelines.


In [ ]:
scenario_path = ensure_path_exists(
    COMPANION_ROOT / "datasets" / "processing" / "sample_processing_scenarios.csv",
    "dataset",
)
scenarios_df = load_processing_scenarios(scenario_path)
scenarios_df[[
    "pipeline_name",
    "source_pattern",
    "freshness_sla_minutes",
    "evaluation_asset",
    "chunking_or_embedding_stage",
    "benchmark_versioned",
]].sort_values("pipeline_name").reset_index(drop=True)

### What to Notice in the Scenario Output

The scenario table matters because it puts evaluation assets and retrieval-preparation stages beside ordinary pipelines. That is the chapter's core move: labels, benchmarks, chunks, and embeddings are all processed assets whose build logic changes downstream model behavior.


## Part 2: Summarize Pressure on the Processing Layer

This summary shows not only latency and replay pressure, but also how many pipelines now affect evaluation or retrieval preparation.


In [ ]:
summary_df = summarize_processing_pressure(scenarios_df)
summary_df


### Interpreting the Pressure Summary

Use the summary to see which constraint is dominating the processing layer. A pipeline estate with many non-replayable inputs or many unversioned benchmark assets is not just inefficient; it is hard to trust after logic changes or incidents.


## Part 3: Build a Processing Plan

The helper attaches recommendations for processing mode, engine choice, replay strategy, and the new evaluation and retrieval guidance fields.


In [ ]:
plan_df = build_processing_plan(scenarios_df)
plan_df[[
    "pipeline_name",
    "recommended_mode",
    "recommended_engine",
    "replay_strategy",
    "evaluation_guidance",
    "retrieval_guidance",
    "primary_risk",
]].sort_values("pipeline_name").reset_index(drop=True)


### Interpreting the Processing Plan

The plan output is where the chapter turns into design guidance. Read across a row and ask whether the chosen mode, engine, replay strategy, and risk statement all tell a coherent story about what the pipeline is actually trying to preserve.


## Part 4: Inspect Evaluation and Benchmark Pipelines

Evaluation sets are processed assets, not static truth. Use this view to ask where leakage can enter, where relabeling changes comparison meaning, and when benchmark generation itself needs a versioned build boundary.

In [ ]:
evaluation_view = plan_df.loc[
    plan_df["evaluation_asset"] == "yes",
    [
        "pipeline_name",
        "recommended_mode",
        "benchmark_versioned",
        "evaluation_guidance",
        "primary_risk",
    ],
].sort_values("pipeline_name").reset_index(drop=True)
evaluation_view


### What the Evaluation View Should Tell You

This output is most useful when it shows that evaluation pipelines are not static reporting jobs. If benchmark versioning or evaluation guidance looks weak here, model comparison will become noisy long before the training code itself appears broken.


## Part 5: Inspect Chunking, Embeddings, and Retrieval Preparation

Chunking and embedding refreshes are transformations whose parameters change downstream behavior. Prompt assembly belongs in the same category: it combines a template, context, policy rules, and tool outputs into one runtime input that can succeed or fail in very ordinary engineering ways.

In [ ]:
retrieval_view = plan_df.loc[
    plan_df["chunking_or_embedding_stage"] == "yes",
    [
        "pipeline_name",
        "recommended_mode",
        "replay_strategy",
        "retrieval_guidance",
        "primary_risk",
    ],
].sort_values("pipeline_name").reset_index(drop=True)
retrieval_view


### What the Retrieval View Should Tell You

Treat the retrieval view as a runtime-behavior warning. Chunking, embedding refreshes, and prompt assembly all change what the model sees, so the output should help you decide whether the rebuild is replayable, versioned, and attached to a clear source snapshot.


## Part 5A: Prompt Assembly as a Transformation

Chapter 5 becomes concrete when prompt assembly is shown as an ordinary transformation step with inputs, ordering, and truncation risk.

In [ ]:
prompt_template = 'Use only the current return policy.\nQuestion: {question}\nContext:\n{context}\nTool output: {tool_output}'
question = 'Can I return an unopened laptop after 35 days?'
retrieved_chunks = [
    {'chunk_id': 'old_policy', 'text': 'Older policy: unopened laptops may be returned within 45 days.'},
    {'chunk_id': 'new_policy', 'text': 'Current policy: unopened laptops may be returned within 30 days.'},
]
tool_output = 'customer_account_status=standard'

def assemble_prompt(template, question, chunks, tool_output, token_budget):
    words = []
    for chunk in chunks:
        words.extend(chunk['text'].split())
    truncated_context = ' '.join(words[:token_budget])
    return template.format(question=question, context=truncated_context, tool_output=tool_output)

full_prompt = assemble_prompt(prompt_template, question, retrieved_chunks, tool_output, token_budget=40)
truncated_prompt = assemble_prompt(prompt_template, question, retrieved_chunks, tool_output, token_budget=9)

print('Full prompt context:')
print(full_prompt)
print('\nPrompt after aggressive truncation:')
print(truncated_prompt)

### What to Notice in the Prompt Assembly Output

The full prompt preserves both policy statements. The truncated prompt keeps only the older one. Nothing about the model changed, but the transformation changed meaning by removing the current policy before inference even started.

## Part 5B: Compare Chunking Strategies

Chunking is another transformation choice that changes meaning rather than merely formatting text.

In [ ]:
chunking_comparison = pd.DataFrame([
    {
        'chunking_strategy': 'fixed_size_chunks',
        'sample_chunk_ids': ['chunk_001', 'chunk_002', 'chunk_003'],
        'dominant_risk': 'policy sentences can split across arbitrary boundaries',
    },
    {
        'chunking_strategy': 'section_aware_chunks',
        'sample_chunk_ids': ['returns_section', 'warranty_section'],
        'dominant_risk': 'fewer chunks, but retrieval depends more on section metadata quality',
    },
])
chunking_comparison

## Part 5C: Check Embedding Lineage

A retrieval rebuild is only reproducible if corpus snapshot, chunking config, embedding model, and index version stay linked.

In [ ]:
embedding_lineage_check = pd.DataFrame([
    {
        'corpus_snapshot': 'support_docs_v14',
        'chunking_config': 'section_chunks_v3',
        'embedding_model_version': 'text-embedding-3-large',
        'vector_index_version': 'support_index_v14_2',
        'lineage_status': 'reconstructable',
    },
    {
        'corpus_snapshot': 'support_docs_live',
        'chunking_config': '',
        'embedding_model_version': 'unknown',
        'vector_index_version': 'support_index_live',
        'lineage_status': 'unsafe for audit or rollback',
    },
])
embedding_lineage_check

## Part 5D: Stress Benchmark Leakage

Leakage is easiest to see when one feature join accidentally uses future information.

In [ ]:
feature_history = pd.DataFrame([
    {'customer_id': 1, 'feature_ts': pd.Timestamp('2026-01-10'), 'sessions_7d': 3},
    {'customer_id': 1, 'feature_ts': pd.Timestamp('2026-01-20'), 'sessions_7d': 9},
])
label_rows = pd.DataFrame([
    {'customer_id': 1, 'label_ts': pd.Timestamp('2026-01-15'), 'converted': 1},
])
correct_value = feature_history.loc[feature_history['feature_ts'] <= label_rows.loc[0, 'label_ts'], 'sessions_7d'].iloc[-1]
incorrect_value = feature_history.sort_values('feature_ts').drop_duplicates('customer_id', keep='last').iloc[0]['sessions_7d']
leakage_comparison = pd.DataFrame([
    {'join_type': 'point_in_time_correct', 'sessions_7d_used': correct_value, 'meaning': 'value known at label time'},
    {'join_type': 'future_leaking_join', 'sessions_7d_used': incorrect_value, 'meaning': 'value only known after the label event'},
])
leakage_comparison

### Why the Leakage Example Matters

The incorrect join produces a stronger-looking feature, but it does so by importing future information into the training example. The pipeline still runs. The meaning is what broke.

## Part 6: Compare the Highest-Risk Pipelines

These are the scenarios most likely to fail because of weak replay, unsafe incremental logic, benchmark drift, stale retrieved context, wrong template selection, malformed prompt variables, or token truncation that removes the very policy text the system depended on.

In [ ]:
high_risk_df = find_high_risk_pipelines(plan_df)
high_risk_df


### How to Read the Processing Recommendations

Treat each high-risk row as a transformation contract that is starting to break. The useful question is not only which pipeline looks risky. It is whether the risk comes from replay weakness, leakage into evaluation, stale retrieval preparation, or confusion between prompt template, prompt instance, context, trace, and final evaluation result.


## Part 6A: Inspect Processing Artifacts

The chapter points to concrete processing artifacts because replay, point-in-time correctness, and chunking policy should be visible outside the scenario table. The snippets below show the companion versions of those contracts.

In [ ]:
artifacts_dir = ensure_path_exists(
    COMPANION_ROOT / "artifacts" / "processing",
    "artifact directory",
)
print("dbt_incremental.sql")
print("-" * 60)
print("\n".join((ensure_path_exists(artifacts_dir / "dbt_incremental.sql", "artifact")).read_text(encoding="utf-8").splitlines()[:18]))
print("\npoint_in_time_join.sql")
print("-" * 60)
print("\n".join((ensure_path_exists(artifacts_dir / "point_in_time_join.sql", "artifact")).read_text(encoding="utf-8").splitlines()[:18]))
print("\nchunking_config.yaml")
print("-" * 60)
print("\n".join((ensure_path_exists(artifacts_dir / "chunking_config.yaml", "artifact")).read_text(encoding="utf-8").splitlines()[:18]))
print("\npipeline_planning.py")
print("-" * 60)
print("\n".join(module_path.read_text(encoding="utf-8").splitlines()[:18]))

## Exercise

### Concept check
- Classify the short incidents in this chapter as point-in-time leakage, semantic transformation failure, retrieval preparation failure, prompt assembly failure, benchmark drift, budget failure, or evidence failure.

### Diagnostic scenario
- Use the laptop return-policy example to explain whether the model failed or the transformation failed, which prompt or context artifact should have been logged, and what pre-release test should have caught the issue.

### Artifact-building exercise
- Build a transformation-lineage record for one feature table, one benchmark set, one retrieval asset, and one prompt instance. Include source snapshot, transform code or configuration, version, validation check, and owner.

### Notebook extension
- Change the token budget in the prompt-assembly example, identify when the answer becomes unsafe, and finish with a short transformation risk report naming the highest-risk transform in the notebook.